In [1]:
# Risking a Southern North Sea gas exploration well
# A hypothetical single-well prospect. all inputs are illustrative 

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
N = 100_000
Z90 = 1.2816

In [3]:
# Volumetrics (BCF). P90 is the low case, P10 the high case
GIIP_P90, GIIP_P10 = 10.0, 40.0
RF_MIN, RF_MODE, RF_MAX = 0.65, 0.75, 0.85

# Costs (GBP m)
WELL_COST = 25.0
COST_MIN, COST_MODE, COST_MAX = 0.90 * WELL_COST, WELL_COST, 1.15 * WELL_COST
SPREAD_RATE = 0.20
NPT_MIN, NPT_MODE, NPT_MAX = 0, 10, 50

# Commercial
PRICE_MEDIAN, PRICE_SIGMA = 70.0, 0.25
OPEX_PER_BCF = 1.0
THERMS_PER_BCF = 10.37e6

# Production and discounting
DISCOUNT, DECLINE, YEARS = 0.10, 0.20, 10

# Chance of success
PLAY = {"Source": 1.0, "Charge": 0.80, "Reservoir": 0.90, "Trap/Seal": 0.55}
COS = float(np.prod(list(PLAY.values())))
print(f"Overall chance of success: {COS:.1%}")

Overall chance of success: 39.6%


In [4]:
mu = (np.log(GIIP_P10) + np.log(GIIP_P90)) / 2
sigma = (np.log(GIIP_P10) - np.log(GIIP_P90)) / (2 * Z90)
print(np.exp(mu), sigma)

20.000000000000007 0.5408451783395327


In [5]:
# 2. Economic Model 
# Reserves = GIIP x recovery factor. 
# Each BCF earns price minus opex. Production declines 20% a year for 10 years and is discounted mid-year. 
# The well cost is spent up front, plus the cost of any extra days lost to NPT.


In [6]:
years = np.arange(YEARS)
share = (1 - DECLINE) ** years
share = share / share.sum()
PV_FACTOR = float((share / (1 + DISCOUNT) ** (years + 0.5)).sum())
print(PV_FACTOR)

0.750886410015419


In [7]:
def well_model(giip, rf, price, base_cost, npt_days):
    reserves = giip * rf
    margin = THERMS_PER_BCF / 1e6 * price / 100 - OPEX_PER_BCF
    cost = base_cost + npt_days * SPREAD_RATE
    undiscounted = reserves * margin - cost
    discounted = PV_FACTOR * reserves * margin - cost
    return undiscounted, discounted, cost

    

In [8]:
## 3. Monte Carlo

In [9]:
inputs = pd.DataFrame({
    "giip": rng.lognormal(mu, sigma, N),
    "rf": rng.triangular(RF_MIN, RF_MODE, RF_MAX, N),
    "price": rng.lognormal(np.log(PRICE_MEDIAN), PRICE_SIGMA, N),
    "base_cost": rng.triangular(COST_MIN, COST_MODE, COST_MAX, N),
    "npt_days": rng.triangular(NPT_MIN, NPT_MODE, NPT_MAX, N),
})

In [10]:
undisc, npv, total_cost = well_model(
    inputs["giip"], inputs["rf"], inputs["price"],
    inputs["base_cost"], inputs["npt_days"])

In [11]:
inputs.head()

,giip,rf,price,base_cost,npt_days
0,23.583258,0.746476,66.682645,24.062621,17.672546
1,11.395994,0.831070,74.875479,23.069266,11.141744
2,30.012387,0.837330,118.928147,24.779951,32.816375
3,33.262551,0.819749,96.210787,27.107794,14.816705
4,6.962400,0.820092,50.377296,23.456773,42.492090


In [12]:
def p_cases(x):
    return pd.Series({"P90": np.percentile(x, 10), "P50": np.percentile(x, 50),
                      "P10": np.percentile(x, 90), "Mean": np.mean(x)})

In [13]:
summary = pd.DataFrame({
    "GIIP (BCF)": p_cases(inputs["giip"]),
    "Reserves (BCF)": p_cases(inputs["giip"] * inputs["rf"]),
    "Total well cost (GBP m)": p_cases(total_cost),
    "Undiscounted net cash flow (GBP m)": p_cases(undisc),
    "Discounted NPV (GBP m)": p_cases(npv),
}).T.round(1)
summary

,P90,P50,P10,Mean
GIIP (BCF),10.0,19.9,40.0,23.1
Reserves (BCF),7.4,14.9,30.1,17.3
Total well cost (GBP m),26.3,29.2,32.9,29.4
Undiscounted net cash flow (GBP m),12.7,63.6,176.1,83.1
Discounted NPV (GBP m),2.1,40.5,124.9,55.1


In [14]:
## Histogram

In [15]:
print(f"Probability NPV < 0, given success: {(npv < 0).mean():.1%}")

Probability NPV < 0, given success: 8.2%


In [16]:
low, high = np.percentile(npv, [0.5, 99.5])
ax.hist(npv, bins=120, range=(low, high), color="#7a9cc6")

NameError: name 'ax' is not defined

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))

low, high = np.percentile(npv, [0.5, 99.5])
ax.hist(npv, bins=120, range=(low, high), color="#7a9cc6")

p = p_cases(npv)
for name, colour in (("P90", "#b03a2e"), ("P50", "#222222"), ("P10", "#1e8449")):
    ax.axvline(p[name], color=colour, lw=1.6)

ax.set_xlabel("Discounted NPV, GBP m")
ax.set_ylabel("Iterations")
ax.set_title("NPV if the well finds gas")
plt.show()

In [ ]:
## Sensitivity

In [ ]:
LABELS = {"giip": "GIIP", "rf": "Recovery factor", "price": "Gas price",
          "base_cost": "Base well cost", "npt_days": "Extra NPT days"}

base = inputs.median()
base_npv = well_model(**base.to_dict())[1]
print("NPV with every input at its median:", round(base_npv, 1))

In [ ]:
rows = []
for col in inputs.columns:
    low, high = inputs[col].quantile([0.10, 0.90])
    n_low = well_model(**{**base.to_dict(), col: low})[1]
    n_high = well_model(**{**base.to_dict(), col: high})[1]
    rank_corr = inputs[col].rank().corr(pd.Series(npv).rank())
    rows.append({"input": LABELS[col], "npv_low": n_low, "npv_high": n_high,
                 "swing": abs(n_high - n_low), "rank_corr": rank_corr})

tornado = pd.DataFrame(rows).sort_values("swing", ascending=False).reset_index(drop=True)
tornado.round(2)

In [ ]:
giip_swing = tornado.loc[tornado["input"] == "GIIP", "swing"].iloc[0]
print(f"Extra days needed to lose as much as the GIIP swing: {giip_swing / SPREAD_RATE:.0f}")
print(f"Upper limit of the NPT distribution used here: {NPT_MAX} days")

In [ ]:
## Torando Chart

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.8))

plot_df = tornado.iloc[::-1].reset_index(drop=True)
for i, r in plot_df.iterrows():
    a, b = sorted([r["npv_low"], r["npv_high"]])
    ax.barh(i, b - a, left=a, color="#7a9cc6")
    ax.text(a - 1, i, f"{a:.0f}", ha="right", va="center", fontsize=9)
    ax.text(b + 1, i, f"{b:.0f}", ha="left", va="center", fontsize=9)

ax.axvline(base_npv, color="#222222", lw=1)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["input"])
vals = plot_df[["npv_low", "npv_high"]].to_numpy().ravel()
ax.set_xlim(vals.min() - 10, vals.max() + 10)
ax.set_xlabel("Discounted NPV, GBP m")
ax.set_title("Which uncertainties move NPV")
plt.show()